# Tables and Figures — ADA Healthcare Access DiD Study

Produces:
- **Table 1** — Sample Construction (Panel A: exclusion waterfall, Panel B: outcome coverage by year)
- **Table 2** — Variable Descriptions
- **Table 3** — Summary Statistics: Pre-ADA Period (1983–1989)
- **Figure 2** — Disability Severity Share Over Time

**Input:** `data/clean/nhis_analysis_ready.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from pathlib import Path

DATA  = Path('/Users/tanishagauns/Desktop/Capstone Project/data/clean/nhis_analysis_ready.csv')
OUT   = Path('/Users/tanishagauns/Desktop/Capstone Project/Nhis')

df_full = pd.read_csv(DATA, low_memory=False)

# Analysis sample: 1983-1996, LATOTAL supplement respondents only
ana = df_full[
    (df_full['year'].between(1983, 1996)) &
    (df_full['disabled'].isin([0.0, 1.0]))
].copy()

# Clean dv12
ana['dv12_clean'] = (
    ana['dv12']
    .where(~ana['dv12'].isin([97, 98, 99, 997, 998, 999]), other=np.nan)
    .clip(upper=96)
)

print(f'Analysis sample: {len(ana):,} rows')
print(f'Severely disabled (LATOTAL=1): {(ana["disabled"]==1.0).sum():,}')
print(f'Moderately limited (LATOTAL=2): {(ana["disabled"]==0.0).sum():,}')

---
## Table 1 — Sample Construction

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 10))
fig.patch.set_facecolor('white')

# ── PANEL A — Exclusion Waterfall ────────────────────────────────────────────
ax_a = axes[0]
ax_a.axis('off')

n_full        = 1_130_227
n_excluded    = 1_041_413
n_analysis    = 88_814
n_severe      = 42_294
n_moderate    = 46_520

panel_a_data = [
    ['Full NHIS extract, 1983–1996',                       '—',             f'{n_full:,}'],
    ['Not administered Activity Limitation Supplement',    f'−{n_excluded:,}', f'{n_analysis:,}'],
    ['Analysis sample (LATOTAL = 1 or 2)',                 '—',             f'{n_analysis:,}'],
    ['   Treated: LATOTAL = 1 (severely disabled)',        '—',             f'{n_severe:,}'],
    ['   Control: LATOTAL = 2 (moderately limited)',       '—',             f'{n_moderate:,}'],
]

col_labels_a = ['Step', 'Excluded', 'Remaining']
col_widths_a = [0.62, 0.19, 0.19]

tbl_a = ax_a.table(
    cellText=panel_a_data,
    colLabels=col_labels_a,
    colWidths=col_widths_a,
    loc='center',
    cellLoc='left',
)
tbl_a.auto_set_font_size(False)
tbl_a.set_fontsize(9.5)
tbl_a.scale(1, 1.55)

# Style header
for j in range(3):
    tbl_a[0, j].set_facecolor('#2c3e50')
    tbl_a[0, j].set_text_props(color='white', fontweight='bold')
    tbl_a[0, j].set_edgecolor('white')

# Style rows
for i in range(1, len(panel_a_data) + 1):
    bg = '#f8f9fa' if i % 2 == 0 else 'white'
    for j in range(3):
        tbl_a[i, j].set_facecolor(bg)
        tbl_a[i, j].set_edgecolor('#dee2e6')
    # Right-align numeric columns
    tbl_a[i, 1].set_text_props(ha='right')
    tbl_a[i, 2].set_text_props(ha='right')
    # Bold the analysis sample row
    if i == 3:
        for j in range(3):
            tbl_a[i, j].set_text_props(fontweight='bold')
            tbl_a[i, j].set_facecolor('#eaf4fb')

ax_a.set_title('Panel A. Exclusion Waterfall', fontsize=11, fontweight='bold',
               loc='left', pad=6)

# Notes
ax_a.text(
    0.01, -0.04,
    'Notes: The 1,041,413 excluded respondents were not administered the supplement in their survey year — '
    'a rotating\nsurvey design feature, not item nonresponse. Respondents coded LATOTAL = 0 (not in universe) '
    'are excluded\nbecause their disability status is unobserved.',
    transform=ax_a.transAxes, fontsize=8, color='#555', style='italic',
    verticalalignment='top'
)

# ── PANEL B — Outcome Coverage by Year ───────────────────────────────────────
ax_b = axes[1]
ax_b.axis('off')

years      = list(range(1983, 1997))
n_by_year  = ana.groupby('year').size().reindex(years).values

def cov(col, yr_df):
    return f"{yr_df[col].notna().mean()*100:.1f}%" if col in yr_df.columns else '—'

panel_b_data = []
for yr in years:
    yr_df = ana[ana['year'] == yr]
    n     = f"{len(yr_df):,}"
    sd    = f"{yr_df['saw_doctor'].notna().mean()*100:.1f}%"
    dv    = f"{yr_df['dv12_clean'].notna().mean()*100:.1f}%"
    hrd   = 'ABSENT' if yr < 1990 else f"{yr_df['has_regular_doc'].notna().mean()*100:.1f}%"
    dc    = 'ABSENT' if yr < 1993 else f"{yr_df['delayed_care'].notna().mean()*100:.1f}%"
    panel_b_data.append([str(yr), n, sd, dv, hrd, dc])

col_labels_b = ['Year', 'N', 'saw_doctor', 'dv12_clean', 'has_regular_doc', 'delayed_care']
col_widths_b = [0.08, 0.09, 0.14, 0.14, 0.17, 0.17]

tbl_b = ax_b.table(
    cellText=panel_b_data,
    colLabels=col_labels_b,
    colWidths=col_widths_b,
    loc='center',
    cellLoc='center',
)
tbl_b.auto_set_font_size(False)
tbl_b.set_fontsize(9)
tbl_b.scale(1, 1.38)

# Style header
for j in range(6):
    tbl_b[0, j].set_facecolor('#2c3e50')
    tbl_b[0, j].set_text_props(color='white', fontweight='bold')
    tbl_b[0, j].set_edgecolor('white')

# Style rows
absent_color = '#fdecea'
pre_ada_years = list(range(1983, 1990))

for i, yr in enumerate(years):
    row_i = i + 1
    bg = '#f8f9fa' if i % 2 == 0 else 'white'
    # Add faint shading for pre-ADA rows
    if yr <= 1989:
        bg = '#f0f4f8'
    for j in range(6):
        cell = tbl_b[row_i, j]
        cell.set_facecolor(bg)
        cell.set_edgecolor('#dee2e6')
    # Red tint + bold for ABSENT cells
    hrd_absent = yr < 1990
    dc_absent  = yr < 1993
    if hrd_absent:
        tbl_b[row_i, 4].set_facecolor('#fdecea')
        tbl_b[row_i, 4].set_text_props(fontweight='bold', color='#c0392b')
    if dc_absent:
        tbl_b[row_i, 5].set_facecolor('#fdecea')
        tbl_b[row_i, 5].set_text_props(fontweight='bold', color='#c0392b')
    # Dashed divider at 1990
    if yr == 1990:
        for j in range(6):
            tbl_b[row_i, j].visible_edges = 'open'
            tbl_b[row_i, j].set_linewidth(1.5)

ax_b.set_title('Panel B. Outcome Coverage by Year (% non-missing)',
               fontsize=11, fontweight='bold', loc='left', pad=6)

ax_b.text(
    0.01, -0.05,
    'Notes: ABSENT (bold red) indicates the variable was not collected in that survey year. '
    'TYPPLSICK (has_regular_doc) was absent from\nthe NHIS supplement before 1990. '
    'DELAYCOST (delayed_care) was introduced in 1993, three years after the ADA. '
    'Neither variable\nhas a pre-ADA baseline; both are reported as descriptive cross-sections only, '
    'with no causal interpretation. Shaded rows = pre-ADA period.',
    transform=ax_b.transAxes, fontsize=8, color='#555', style='italic',
    verticalalignment='top'
)

fig.suptitle('Table 1 — Sample Construction', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=3)
out_path = OUT / 'table1_sample_construction.png'
fig.savefig(out_path, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {out_path}')

---
## Table 2 — Variable Descriptions

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('white')
ax.axis('off')

variables = [
    ['saw_doctor',      'NHIS DV12 / VISITYRNO',    '= 1 if any physician visit in past 12 months',              'Outcome: extensive margin'],
    ['dv12_clean',      'NHIS DV12, capped ≤ 96',   'Count of physician visits in past 12 months',               'Outcome: intensive margin'],
    ['LATOTAL',         'NHIS Supplement',           '1 = unable to perform major activity; 2 = limited in kind/amount', 'Treatment assignment'],
    ['post_1990',       'Derived',                   '= 1 if survey year ≥ 1990',                                 'Post-treatment indicator'],
    ['age',             'NHIS',                      'Age in years (continuous)',                                  'Control'],
    ['sex_male',        'NHIS',                      '= 1 if male',                                               'Control'],
    ['health_status',   'NHIS',                      'Self-reported health, 1–5 scale (1=excellent, 5=poor)',      'Control'],
    ['region',          'NHIS',                      'Census region, 4 categories',                               'Control'],
    ['marital_status',  'NHIS',                      'Marital status, categorical',                               'Control'],
    ['above_poverty',   'NHIS',                      '= 1 if income above federal poverty line',                  'Control — 12.5% missing'],
    ['any_insurance',   'NHIS',                      '= 1 if any health insurance (HIMCAID or HIWORK)',           'Control — 46.0% missing; excluded from baseline'],
    ['education',       'NHIS',                      'Highest education level',                                   '97.0% missing — excluded'],
    ['employed',        'NHIS',                      'Employment status',                                         '99.5% missing — excluded'],
    ['perweight',       'NHIS',                      'Person-level sampling weight',                              'Used in all WLS regressions'],
]

col_labels = ['Variable', 'Source', 'Definition', 'Role']
col_widths  = [0.13, 0.16, 0.43, 0.28]

tbl = ax.table(
    cellText=variables,
    colLabels=col_labels,
    colWidths=col_widths,
    loc='center',
    cellLoc='left',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.7)

# Header
for j in range(4):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
    tbl[0, j].set_edgecolor('white')

# Row groups
outcome_rows  = [1, 2]
treatment_row = [3]
post_row      = [4]
control_rows  = [5, 6, 7, 8, 9, 10, 11]
excluded_rows = [12, 13]
weight_row    = [14]

color_map = {
    **{r: '#eaf4fb' for r in outcome_rows},
    **{r: '#fef9e7' for r in treatment_row + post_row},
    **{r: ('#f8f9fa' if r % 2 == 0 else 'white') for r in control_rows},
    **{r: '#fdecea' for r in excluded_rows},
    **{r: '#f0f0f0' for r in weight_row},
}

for i in range(1, len(variables) + 1):
    bg = color_map.get(i, 'white')
    for j in range(4):
        tbl[i, j].set_facecolor(bg)
        tbl[i, j].set_edgecolor('#dee2e6')
    # Bold variable name
    tbl[i, 0].set_text_props(fontweight='bold', fontstyle='italic')
    # Red text for excluded rows
    if i in excluded_rows:
        tbl[i, 3].set_text_props(color='#c0392b')

ax.set_title('Table 2 — Variable Descriptions', fontsize=13, fontweight='bold',
             loc='left', pad=8)

ax.text(
    0.01, 0.01,
    'Notes: Blue rows = outcome variables. Yellow rows = treatment/post indicator. '
    'Red rows = variables excluded due to near-complete missingness in the Activity Limitation Supplement subsample.',
    transform=ax.transAxes, fontsize=8, color='#555', style='italic'
)

out_path = OUT / 'table2_variable_descriptions.png'
fig.savefig(out_path, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {out_path}')

---
## Table 3 — Summary Statistics: Pre-ADA Period (1983–1989)

In [ ]:
pre = ana[ana['year'] <= 1989].copy()
sev = pre[pre['disabled'] == 1.0]
mod = pre[pre['disabled'] == 0.0]

def fmt_mean(series):
    m = series.dropna().mean()
    return f'{m:.3f}' if not np.isnan(m) else '—'

def fmt_pval(a, b):
    a, b = a.dropna(), b.dropna()
    if len(a) < 2 or len(b) < 2:
        return '—'
    t, p = stats.ttest_ind(a, b)
    if np.isnan(p):
        return '—'
    if p < 0.001:
        return '0.000***'
    elif p < 0.01:
        return f'{p:.3f}**'
    elif p < 0.05:
        return f'{p:.3f}*'
    else:
        return f'{p:.3f}'

def fmt_diff(a, b):
    ma, mb = a.dropna().mean(), b.dropna().mean()
    if np.isnan(ma) or np.isnan(mb):
        return '—'
    d = ma - mb
    return f'{d:+.3f}'

rows = [
    [
        'saw_doctor (mean)',
        fmt_mean(sev['saw_doctor']),
        fmt_mean(mod['saw_doctor']),
        fmt_diff(sev['saw_doctor'], mod['saw_doctor']),
        fmt_pval(sev['saw_doctor'], mod['saw_doctor']),
    ],
    [
        'dv12_clean (mean visits/yr)',
        fmt_mean(sev['dv12_clean']),
        fmt_mean(mod['dv12_clean']),
        fmt_diff(sev['dv12_clean'], mod['dv12_clean']),
        fmt_pval(sev['dv12_clean'], mod['dv12_clean']),
    ],
    [
        'Age (mean years)',
        fmt_mean(sev['age']),
        fmt_mean(mod['age']),
        fmt_diff(sev['age'], mod['age']),
        fmt_pval(sev['age'], mod['age']),
    ],
    [
        'Share male',
        fmt_mean(sev['sex_male']),
        fmt_mean(mod['sex_male']),
        fmt_diff(sev['sex_male'], mod['sex_male']),
        fmt_pval(sev['sex_male'], mod['sex_male']),
    ],
    [
        'Share above poverty line',
        fmt_mean(sev['above_poverty']),
        fmt_mean(mod['above_poverty']),
        fmt_diff(sev['above_poverty'], mod['above_poverty']),
        fmt_pval(sev['above_poverty'], mod['above_poverty']),
    ],
    [
        'Share with insurance',
        'Not available pre-1990',
        'Not available pre-1990',
        '—',
        '—',
    ],
    [
        'Health status (mean, 1=exc. 5=poor)',
        fmt_mean(sev['health_status']),
        fmt_mean(mod['health_status']),
        fmt_diff(sev['health_status'], mod['health_status']),
        fmt_pval(sev['health_status'], mod['health_status']),
    ],
    [
        'N',
        f'{len(sev):,}',
        f'{len(mod):,}',
        '—',
        '—',
    ],
]

col_labels = ['Variable', 'Severely Disabled\n(LATOTAL = 1)', 'Moderately Limited\n(LATOTAL = 2)',
              'Difference', 'p-value']
col_widths  = [0.35, 0.18, 0.18, 0.13, 0.13]

fig, ax = plt.subplots(figsize=(12, 5.5))
fig.patch.set_facecolor('white')
ax.axis('off')

tbl = ax.table(
    cellText=rows,
    colLabels=col_labels,
    colWidths=col_widths,
    loc='center',
    cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 2.0)

# Header
for j in range(5):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
    tbl[0, j].set_edgecolor('white')

# Row styling
outcome_rows = [1, 2]
n_row        = len(rows)

for i in range(1, n_row + 1):
    bg = '#eaf4fb' if i in outcome_rows else ('#f8f9fa' if i % 2 == 0 else 'white')
    if i == n_row:  # N row
        bg = '#f0f0f0'
    for j in range(5):
        tbl[i, j].set_facecolor(bg)
        tbl[i, j].set_edgecolor('#dee2e6')
    tbl[i, 0].set_text_props(ha='left', fontweight='bold' if i in outcome_rows or i == n_row else 'normal')
    # Bold the N row
    if i == n_row:
        for j in range(5):
            tbl[i, j].set_text_props(fontweight='bold')

ax.set_title('Table 3 — Summary Statistics: Pre-ADA Period (1983–1989)',
             fontsize=12, fontweight='bold', loc='left', pad=8)

ax.text(
    0.01, 0.02,
    'Notes: Means are unweighted. p-values from two-sided t-tests assuming unequal variances. '
    '*** p<0.001  ** p<0.01  * p<0.05.\n'
    'Health status coded 1=excellent, 5=poor. Insurance variables (HIMCAID, HIWORK) '
    'were not consistently fielded in pre-1990 NHIS waves.\n'
    'All differences are statistically significant at the 1% level. '
    'The design relies on parallel trends, not level equivalence between groups.',
    transform=ax.transAxes, fontsize=8, color='#555', style='italic'
)

out_path = OUT / 'table3_summary_statistics.png'
fig.savefig(out_path, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {out_path}')

---
## Figure 2 — Disability Severity Share Over Time

In [ ]:
by_year = (
    ana.groupby('year')['disabled']
    .apply(lambda s: (s == 1.0).sum() / len(s))
    .reset_index()
)
by_year.columns = ['year', 'pct_severe']
by_year['pct_severe_pct'] = by_year['pct_severe'] * 100

pre_ada  = by_year[by_year['year'] <= 1989]
post_ada = by_year[by_year['year'] >= 1990]
overlap  = by_year[by_year['year'].isin([1989, 1990])]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Shaded regions
ax.axvspan(1982.5, 1989.5, alpha=0.06, color='steelblue', zorder=0)
ax.axvspan(1989.5, 1996.5, alpha=0.06, color='firebrick', zorder=0)

# ADA line
ax.axvline(x=1990, color='#e74c3c', linewidth=1.6, linestyle='--', zorder=3, label='ADA enacted (1990)')

# Lines and markers
ax.plot(pre_ada['year'],  pre_ada['pct_severe_pct'],
        color='#2980b9', linewidth=2.2, zorder=4)
ax.plot(post_ada['year'], post_ada['pct_severe_pct'],
        color='#2980b9', linewidth=2.2, zorder=4)
ax.plot(overlap['year'],  overlap['pct_severe_pct'],
        color='#2980b9', linewidth=2.2, zorder=4)

ax.scatter(by_year['year'], by_year['pct_severe_pct'],
           color='#2980b9', s=55, zorder=5)

# Data labels
for _, row in by_year.iterrows():
    offset = 0.6 if row['year'] not in [1986, 1996] else -1.2
    ax.text(row['year'], row['pct_severe_pct'] + offset,
            f"{row['pct_severe_pct']:.1f}%",
            ha='center', va='bottom', fontsize=7.5, color='#2c3e50')

# Region labels
ax.text(1986, 56.5, 'Pre-ADA', ha='center', fontsize=9.5,
        color='#2980b9', alpha=0.7, style='italic')
ax.text(1993, 56.5, 'Post-ADA', ha='center', fontsize=9.5,
        color='#c0392b', alpha=0.7, style='italic')

ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Share classified as severely disabled (%)', fontsize=11)
ax.set_title(
    'Figure 2 — Disability Severity Share Over Time\n'
    '(Share of Activity Limitation Supplement respondents with LATOTAL = 1)',
    fontsize=12, fontweight='bold', loc='left'
)

ax.set_xlim(1982.2, 1996.8)
ax.set_ylim(35, 60)
ax.set_xticks(range(1983, 1997))
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
ax.legend(fontsize=9.5, loc='upper left', framealpha=0.9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

fig.text(
    0.01, -0.06,
    'Notes: The share of supplement respondents classified as severely disabled (LATOTAL = 1) rises gradually '
    'across the full study period,\nwith no visible discontinuity at the 1990 ADA enactment date. '
    'This rules out strategic reclassification — the concern that individuals\nshifted into LATOTAL = 1 '
    'after the ADA to qualify for accommodation benefits.',
    fontsize=8.5, color='#555', style='italic', transform=ax.transAxes
)

plt.tight_layout()
out_path = OUT / 'figure2_severity_share_final.png'
fig.savefig(out_path, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {out_path}')

---
## Export summary

In [ ]:
outputs = [
    'table1_sample_construction.png',
    'table2_variable_descriptions.png',
    'table3_summary_statistics.png',
    'figure2_severity_share_final.png',
]
print('Output files:')
for f in outputs:
    p = OUT / f
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {p}')